# Importing the libaries

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score
from google.colab import files


# Pandas
- mounting the dataset
- making copy of the data
- first 5 rows

In [3]:
path = "/content/drive/MyDrive/dataset/data.csv"

df_original = pd.read_csv(path)
df = df_original.copy()
print("the original data", df_original.shape)
print("the working data", df.shape)
df.head()

the original data (29332, 87)
the working data (29332, 87)


,android.permission.GET_ACCOUNTS,com.sonyericsson.home.permission.BROADCAST_BADGE,android.permission.READ_PROFILE,android.permission.MANAGE_ACCOUNTS,android.permission.WRITE_SYNC_SETTINGS,android.permission.READ_EXTERNAL_STORAGE,android.permission.RECEIVE_SMS,com.android.launcher.permission.READ_SETTINGS,android.permission.WRITE_SETTINGS,com.google.android.providers.gsf.permission.READ_GSERVICES,...,com.android.launcher.permission.UNINSTALL_SHORTCUT,com.sec.android.iap.permission.BILLING,com.htc.launcher.permission.UPDATE_SHORTCUT,com.sec.android.provider.badge.permission.WRITE,android.permission.ACCESS_NETWORK_STATE,com.google.android.finsky.permission.BIND_GET_INSTALL_REFERRER_SERVICE,com.huawei.android.launcher.permission.READ_SETTINGS,android.permission.READ_SMS,android.permission.PROCESS_INCOMING_CALLS,Result
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0


# Data Info and cleaning
- the shape of the data

In [4]:
print("number of row and colums", df.shape)
#print("\n", "#" * 50 , "\n")
#df.info()
#print("\n", "#" * 50 , "\n")
#print(df.columns)
#print("\n", "#" * 50 , "\n")
#print(df.describe())
#print("\n", "#" * 50 , "\n")
#df['Result'].value_counts()



number of row and colums (29332, 87)


## Data Cleaning
- droping null values
- removing duplicates values


In [5]:
print("the number of null values in the dataset",df_original.isnull().sum().sum())
df.dropna(inplace=True)
print("the number of duplicates in the dataset", df_original.duplicated().sum())
df.drop_duplicates(inplace=True)

df.isnull().sum().sort_values(ascending=False).head(10)


the number of null values in the dataset 0
the number of duplicates in the dataset 21841


,0
android.permission.GET_ACCOUNTS,0
com.sonyericsson.home.permission.BROADCAST_BADGE,0
android.permission.READ_PROFILE,0
android.permission.MANAGE_ACCOUNTS,0
android.permission.WRITE_SYNC_SETTINGS,0
android.permission.READ_EXTERNAL_STORAGE,0
android.permission.RECEIVE_SMS,0
com.android.launcher.permission.READ_SETTINGS,0
android.permission.WRITE_SETTINGS,0
com.google.android.providers.gsf.permission.READ_GSERVICES,0


In [6]:
print("the original data", df_original.shape)
print("the working data", df.shape)

the original data (29332, 87)
the working data (7491, 87)


In [7]:
print(df['Result'].value_counts())
print("#" * 50)
df.dtypes.value_counts()



Result
0    4867
1    2624
Name: count, dtype: int64
##################################################


,count
int64,87


# **Scikit learn**
- spliting the data into train and split:
x = permissions and y = Result
- why the split ? the modul analyze the permissions (X) and learn what to expect (y) to be, so when the app sent a list of an app permissions the modul will have the ability of predict wether the app is begine or malware
- Y = series in python meaning one column and multiable rows

In [8]:
X = df.drop(columns=['Result'])
y = df['Result']

print(X.shape)
print(y.shape)
print(y.head())


(7491, 86)
(7491,)
0    0
1    0
2    0
3    0
4    0
Name: Result, dtype: int64


## **train_test_split**
- Training set: نستخدمها لتدريب النموذج
- Testing set: نستخدمها لتقييم أداء النموذج على بيانات ما شافها قبلًا، حتى نتأكد أن النموذج ما تعلم حفظ البيانات (Overfitting).

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(5992, 86)
(1499, 86)
(5992,)
(1499,)


# **The model object**

## DecisionTreeClassifier

### the model

In [10]:
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
model.predict(X_test)
y_pred = model.predict(X_test)

### Accuracy, CM, recall and precision

In [11]:
print(y_pred[:10])
print(y_test.values[:10])

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

cm = confusion_matrix(y_test, y_pred)
print(cm)

recall = recall_score(y_test, y_pred)
print("Recall:", recall)

[1 0 0 0 1 0 0 1 0 0]
[0 0 0 1 1 0 0 1 0 0]
Accuracy: 0.9039359573048699
[[943  72]
 [ 72 412]]
Recall: 0.8512396694214877


## Random Forest

### first try

* **100 trees** → Good accuracy/speed balance; performance plateaus after this.
* **max_depth=None** → Deep trees capture complex, non-linear malware patterns.
* **n_jobs=-1** → Uses all CPU cores; trees train in parallel.


In [12]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(y_pred_rf[:10])
print(y_test.values[:10])

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[0 0 0 1 1 0 0 1 0 0]
[0 0 0 1 1 0 0 1 0 0]
[[970  45]
 [ 47 437]]
              precision    recall  f1-score   support

           0       0.95      0.96      0.95      1015
           1       0.91      0.90      0.90       484

    accuracy                           0.94      1499
   macro avg       0.93      0.93      0.93      1499
weighted avg       0.94      0.94      0.94      1499



In [13]:
y_proba = rf.predict_proba(X_test)[:, 1]
threshold = 0.3
y_pred_custom = (y_proba >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))


[[906 109]
 [ 27 457]]
              precision    recall  f1-score   support

           0       0.97      0.89      0.93      1015
           1       0.81      0.94      0.87       484

    accuracy                           0.91      1499
   macro avg       0.89      0.92      0.90      1499
weighted avg       0.92      0.91      0.91      1499



### why 27 malware has ran

- safe them as an index
- look at thire features

In [14]:
# فهارس FN
fn_idx = np.where((y_test == 1) & (y_pred_custom == 0))[0]
len(fn_idx)


27

In [15]:
importances = rf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

feature_importance_df.head(10)

,Feature,Importance
21,android.permission.READ_PHONE_STATE,0.195079
39,com.google.android.c2dm.permission.RECEIVE,0.150187
17,com.android.launcher.permission.INSTALL_SHORTCUT,0.093702
65,android.permission.RECEIVE_BOOT_COMPLETED,0.041386
5,android.permission.READ_EXTERNAL_STORAGE,0.039799
57,android.permission.SYSTEM_ALERT_WINDOW,0.033574
63,com.android.vending.BILLING,0.031127
11,android.permission.GET_TASKS,0.029258
53,android.permission.ACCESS_COARSE_LOCATION,0.025039
66,android.permission.WAKE_LOCK,0.022317


In [16]:
fn_samples = X_test.iloc[fn_idx]
# print(fn_samples)
fn_samples.mean().sort_values(ascending=False).head(10)


,0
android.permission.INTERNET,1.000000
android.permission.ACCESS_NETWORK_STATE,0.925926
android.permission.WRITE_EXTERNAL_STORAGE,0.814815
android.permission.READ_PHONE_STATE,0.629630
android.permission.ACCESS_WIFI_STATE,0.592593
android.permission.WAKE_LOCK,0.370370
android.permission.VIBRATE,0.370370
android.permission.GET_ACCOUNTS,0.296296
com.google.android.c2dm.permission.RECEIVE,0.259259
com.android.vending.BILLING,0.222222


In [17]:
print(df.columns.tolist())

['android.permission.GET_ACCOUNTS', 'com.sonyericsson.home.permission.BROADCAST_BADGE', 'android.permission.READ_PROFILE', 'android.permission.MANAGE_ACCOUNTS', 'android.permission.WRITE_SYNC_SETTINGS', 'android.permission.READ_EXTERNAL_STORAGE', 'android.permission.RECEIVE_SMS', 'com.android.launcher.permission.READ_SETTINGS', 'android.permission.WRITE_SETTINGS', 'com.google.android.providers.gsf.permission.READ_GSERVICES', 'android.permission.DOWNLOAD_WITHOUT_NOTIFICATION', 'android.permission.GET_TASKS', 'android.permission.WRITE_EXTERNAL_STORAGE', 'android.permission.RECORD_AUDIO', 'com.huawei.android.launcher.permission.CHANGE_BADGE', 'com.oppo.launcher.permission.READ_SETTINGS', 'android.permission.CHANGE_NETWORK_STATE', 'com.android.launcher.permission.INSTALL_SHORTCUT', 'android.permission.android.permission.READ_PHONE_STATE', 'android.permission.CALL_PHONE', 'android.permission.WRITE_CONTACTS', 'android.permission.READ_PHONE_STATE', 'com.samsung.android.providers.context.permi

### adding the new features

In [18]:
# permission_columns = df.drop('Result', axis=1).columns # it did not worked
permission_columns = [col for col in df.columns if 'permission' in col.lower()]

sensitive_perms = [
    'android.permission.READ_PHONE_STATE',
    'android.permission.READ_EXTERNAL_STORAGE',
    'android.permission.WRITE_EXTERNAL_STORAGE'
]

network_perms = [
    'android.permission.INTERNET',
    'android.permission.ACCESS_NETWORK_STATE',
    'android.permission.ACCESS_WIFI_STATE',
    'com.google.android.c2dm.permission.RECEIVE'
]

persistence_perms = [
    'android.permission.RECEIVE_BOOT_COMPLETED',
    'android.permission.WAKE_LOCK',
    'android.permission.SYSTEM_ALERT_WINDOW'
]

binary_permissions = (df[permission_columns] > 0).astype(int)

df['number_of_permissions'] = binary_permissions.sum(axis=1)

df['sensitive_permission_count'] = binary_permissions[sensitive_perms].sum(axis=1)

df['network_permission_ratio'] = (
    binary_permissions[network_perms].sum(axis=1)
    / df['number_of_permissions'].replace(0,1)
)

df['persistence_permission_count'] = binary_permissions[persistence_perms].sum(axis=1)


df['low_perm_high_net'] = (
    (df['number_of_permissions'] <= 8) &
    (df['network_permission_ratio'] > 0.2)
).astype(int)


df['stealth_data_access'] = (
    (df['sensitive_permission_count'] >= 1) &
    (df['persistence_permission_count'] == 0)
).astype(int)

df['minimal_malware_pattern'] = (
    (df['number_of_permissions'] < 10) &
    (df['sensitive_permission_count'] >= 1)
).astype(int)


df[['number_of_permissions',
    'sensitive_permission_count',
    'network_permission_ratio',
    'persistence_permission_count',
    'low_perm_high_net',
    'stealth_data_access',
    'minimal_malware_pattern']].head(10)



# df['sms_and_network'] = (
#     binary_permissions['android.permission.READ_SMS'] &
#     binary_permissions['android.permission.INTERNET']
# ).astype(int)

# df['sensitive_ratio'] = (
#     df['sensitive_permission_count'] /
#     df['number_of_permissions'].replace(0,1)
# )


# system_perms = [col for col in permission_columns if 'SYSTEM' in col]
# df['system_permission_count'] = binary_permissions[system_perms].sum(axis=1)


# df[['number_of_permissions',
#     'sensitive_permission_count',
#     'network_permission_ratio',
#     'persistence_permission_count',
#     'sms_and_network',
#     'sensitive_ratio',
#     'system_permission_count']].head(30)

,number_of_permissions,sensitive_permission_count,network_permission_ratio,persistence_permission_count,low_perm_high_net,stealth_data_access,minimal_malware_pattern
0,0,0,0.000000,0,0,0,0
1,7,2,0.428571,1,1,0,1
2,6,1,0.500000,1,1,0,1
3,3,0,1.000000,0,1,0,0
4,5,0,0.400000,1,1,0,0
5,7,2,0.428571,0,1,1,1
6,6,2,0.500000,0,1,1,1
7,13,2,0.307692,2,0,0,0
8,2,0,1.000000,0,1,0,0
9,23,3,0.173913,1,0,0,0


In [19]:
# for permission_columns = [col for col in df.columns if 'permission' in col.lower()]
df[['number_of_permissions','network_permission_ratio']].describe()


,number_of_permissions,network_permission_ratio
count,7491.000000,7491.000000
mean,11.880123,0.299626
std,6.468867,0.133284
min,0.000000,0.000000
25%,8.000000,0.200000
50%,10.000000,0.272727
75%,14.000000,0.375000
max,63.000000,1.000000


### Random forest second model

In [20]:
X = df.drop('Result', axis=1)
y = df['Result']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


rf_new = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42)

rf_new.fit(X_train, y_train)

y_pred_new = rf_new.predict(X_test)


print(confusion_matrix(y_test, y_pred_new))
print(classification_report(y_test, y_pred_new))

[[911  63]
 [ 48 477]]
              precision    recall  f1-score   support

           0       0.95      0.94      0.94       974
           1       0.88      0.91      0.90       525

    accuracy                           0.93      1499
   macro avg       0.92      0.92      0.92      1499
weighted avg       0.93      0.93      0.93      1499



In [21]:
y_prob_new = rf_new.predict_proba(X_test)[:,1]

threshold = 0.3
y_pred_custom = (y_prob_new >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[868 106]
 [ 27 498]]
              precision    recall  f1-score   support

           0       0.97      0.89      0.93       974
           1       0.82      0.95      0.88       525

    accuracy                           0.91      1499
   macro avg       0.90      0.92      0.91      1499
weighted avg       0.92      0.91      0.91      1499



In [22]:
print(df.columns.tolist())
print("\nNew features exist?")
print(all(col in df.columns for col in [
    'number_of_permissions',
    'sensitive_permission_count',
    'network_permission_ratio',
    'persistence_permission_count'
]))

['android.permission.GET_ACCOUNTS', 'com.sonyericsson.home.permission.BROADCAST_BADGE', 'android.permission.READ_PROFILE', 'android.permission.MANAGE_ACCOUNTS', 'android.permission.WRITE_SYNC_SETTINGS', 'android.permission.READ_EXTERNAL_STORAGE', 'android.permission.RECEIVE_SMS', 'com.android.launcher.permission.READ_SETTINGS', 'android.permission.WRITE_SETTINGS', 'com.google.android.providers.gsf.permission.READ_GSERVICES', 'android.permission.DOWNLOAD_WITHOUT_NOTIFICATION', 'android.permission.GET_TASKS', 'android.permission.WRITE_EXTERNAL_STORAGE', 'android.permission.RECORD_AUDIO', 'com.huawei.android.launcher.permission.CHANGE_BADGE', 'com.oppo.launcher.permission.READ_SETTINGS', 'android.permission.CHANGE_NETWORK_STATE', 'com.android.launcher.permission.INSTALL_SHORTCUT', 'android.permission.android.permission.READ_PHONE_STATE', 'android.permission.CALL_PHONE', 'android.permission.WRITE_CONTACTS', 'android.permission.READ_PHONE_STATE', 'com.samsung.android.providers.context.permi

### Random forest third model

In [23]:
rf_tuned = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=3,
    max_features='sqrt',
    # class_weight={0:1,1:2},
    class_weight={0:1, 1:5},
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)

y_pred_tuned = rf_tuned.predict(X_test)


print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))


[[870 104]
 [ 19 506]]
              precision    recall  f1-score   support

           0       0.98      0.89      0.93       974
           1       0.83      0.96      0.89       525

    accuracy                           0.92      1499
   macro avg       0.90      0.93      0.91      1499
weighted avg       0.93      0.92      0.92      1499



In [24]:
y_prob_tuned = rf_tuned.predict_proba(X_test)[:,1]

threshold = 0.3

y_pred_custom = (y_prob_tuned >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))


[[760 214]
 [  7 518]]
              precision    recall  f1-score   support

           0       0.99      0.78      0.87       974
           1       0.71      0.99      0.82       525

    accuracy                           0.85      1499
   macro avg       0.85      0.88      0.85      1499
weighted avg       0.89      0.85      0.86      1499



In [25]:
print(df.columns.tolist())
print("\nNew features exist?")
print(all(col in df.columns for col in [
    'number_of_permissions',
    'sensitive_permission_count',
    'network_permission_ratio',
    'persistence_permission_count',
    'low_perm_high_net',
    'stealth_data_access',
    'minimal_malware_pattern'
]))


['android.permission.GET_ACCOUNTS', 'com.sonyericsson.home.permission.BROADCAST_BADGE', 'android.permission.READ_PROFILE', 'android.permission.MANAGE_ACCOUNTS', 'android.permission.WRITE_SYNC_SETTINGS', 'android.permission.READ_EXTERNAL_STORAGE', 'android.permission.RECEIVE_SMS', 'com.android.launcher.permission.READ_SETTINGS', 'android.permission.WRITE_SETTINGS', 'com.google.android.providers.gsf.permission.READ_GSERVICES', 'android.permission.DOWNLOAD_WITHOUT_NOTIFICATION', 'android.permission.GET_TASKS', 'android.permission.WRITE_EXTERNAL_STORAGE', 'android.permission.RECORD_AUDIO', 'com.huawei.android.launcher.permission.CHANGE_BADGE', 'com.oppo.launcher.permission.READ_SETTINGS', 'android.permission.CHANGE_NETWORK_STATE', 'com.android.launcher.permission.INSTALL_SHORTCUT', 'android.permission.android.permission.READ_PHONE_STATE', 'android.permission.CALL_PHONE', 'android.permission.WRITE_CONTACTS', 'android.permission.READ_PHONE_STATE', 'com.samsung.android.providers.context.permi

### Feature Selection

In [26]:
importances = rf_tuned.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print(feat_imp.head(20))
print(feat_imp.tail(20))


android.permission.READ_PHONE_STATE                           0.218443
com.google.android.c2dm.permission.RECEIVE                    0.196387
com.android.launcher.permission.INSTALL_SHORTCUT              0.050508
network_permission_ratio                                      0.044006
com.android.vending.BILLING                                   0.043718
android.permission.READ_EXTERNAL_STORAGE                      0.040388
sensitive_permission_count                                    0.037551
number_of_permissions                                         0.028887
android.permission.RECEIVE_BOOT_COMPLETED                     0.019624
com.google.android.providers.gsf.permission.READ_GSERVICES    0.019544
android.permission.SYSTEM_ALERT_WINDOW                        0.018622
android.permission.CAMERA                                     0.016162
persistence_permission_count                                  0.014809
android.permission.ACCESS_COARSE_LOCATION                     0.013815
androi

In [27]:
low_feats = feat_imp[feat_imp < 0.001].index

X_train_red = X_train.drop(columns=low_feats)
X_test_red  = X_test.drop(columns=low_feats)

rf_tuned.fit(X_train_red, y_train)


RandomForestClassifier(class_weight={0: 1, 1: 5}, min_samples_leaf=3,
                       n_estimators=600, n_jobs=-1, random_state=42)

### RF 4 40 features

In [28]:

# ترتيب الأعمدة حسب الأهمية (تنازلي)
feat_imp_sorted = feat_imp.sort_values(ascending=False)

# اختيار أعلى 40 feature
top_40_features = feat_imp_sorted.head(40).index

# إنشاء dataset جديد فقط بأعلى 40 feature
X_train_40 = X_train[top_40_features]
X_test_40  = X_test[top_40_features]

rf_forty = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=4,
    max_features='sqrt',
    # class_weight={0:1,1:2},
    class_weight={0:1, 1:5},
    random_state=42,
    n_jobs=-1
)

# إعادة تدريب النموذج
rf_forty.fit(X_train_40, y_train)

# توقع الاحتمالات
y_prob = rf_forty.predict_proba(X_test_40)[:,1]

# تطبيق threshold
y_pred = (y_prob >= 0.25).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[736 238]
 [  5 520]]
              precision    recall  f1-score   support

           0       0.99      0.76      0.86       974
           1       0.69      0.99      0.81       525

    accuracy                           0.84      1499
   macro avg       0.84      0.87      0.83      1499
weighted avg       0.89      0.84      0.84      1499



In [29]:
y_prob_forty = rf_forty.predict_proba(X_test_40)[:,1]

threshold = 0.25

y_pred_custom = (y_prob_forty >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))


[[736 238]
 [  5 520]]
              precision    recall  f1-score   support

           0       0.99      0.76      0.86       974
           1       0.69      0.99      0.81       525

    accuracy                           0.84      1499
   macro avg       0.84      0.87      0.83      1499
weighted avg       0.89      0.84      0.84      1499



# saving the model

In [30]:
final_model = {
    "model": rf_forty,
    "threshold": 0.3,
    "features": X_train.columns.tolist()
}

with open("malware_rf_final.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open("selected_features.pkl", "wb") as f:
    pickle.dump(top_40_features, f)


print("Final model saved successfully ✅")


Final model saved successfully ✅


In [33]:
files.download("malware_rf_final.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
# with open("malware_rf_final.pkl", "rb") as f:
#     model_test = pickle.load(f)

# with open("selected_features.pkl", "rb") as f:
#     features_test = pickle.load(f)

# print(len(features_test))

# NumPy Function



##  help us to learn what is the correct order of the conftion matrixes
  [[TN  FP]

  [FN  TP]]

In [ ]:
def extract_confusion_elements(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))

    return TP, FN, FP, TN

TP, FN, FP, TN = extract_confusion_elements(y_test, y_pred_rf)

print("TP:", TP)
print("FN:", FN)
print("FP:", FP)
print("TN:", TN)

recall = TP / (TP + FN)
precision = TP / (TP + FP)

print("Recall:", recall)
print("Precision:", precision)



# ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

## Why Visualization?

Visualization helps us:
- Understand label distribution
- Detect class imbalance
- Identify important permissions
- Support feature selection decisions


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")


In [ ]:
plt.figure(figsize=(6,4))
plt.plot([1, 2, 3, 4], [10, 20, 15, 25])
plt.title("Simple Plot Example")
plt.xlabel("X Axis")
plt.ylabel("Y Axis")
plt.show()


In [ ]:
sns.countplot(x=['A', 'B', 'A', 'C', 'B', 'A'])
plt.title("Seaborn Count Plot Example")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Result', data=df)
plt.title("Label Distribution")
plt.show()


In [ ]:
permissions = df.drop(columns=['Result'])

top_permissions = permissions.sum().sort_values(ascending=False).head(15)

plt.figure(figsize=(10,5))
top_permissions.plot(kind='bar')
plt.title("Top 15 Most Frequent Permissions")
plt.ylabel("Count")
plt.show()


In [ ]:
malware = df[df['Result'] == 1]
benign = df[df['Result'] == 0]

malware_sum = malware.drop(columns=['Result']).sum()
benign_sum = benign.drop(columns=['Result']).sum()

compare_df = pd.DataFrame({
    'Malware': malware_sum,
    'Benign': benign_sum
}).sort_values(by='Malware', ascending=False).head(10)

compare_df.plot(kind='bar', figsize=(10,5))
plt.title("Permissions Comparison")
plt.show()
